In [7]:
import requests
import dotenv
import os
import pprint

dotenv.load_dotenv()

True

In [8]:
auth_server_url = "https://sso.slf.ch/auth"
auth_realm = "SLF"
auth_client_id = "snowprofiler"
auth_username = "yannik.werner@slf.ch"
auth_password = os.getenv("SNOWPROFILER_PASSWORD")

snowprofiler_api_url = "https://snowprofiler-api.int.slf.ch"

In [9]:
def get_keycloak_token():
    r = requests.post(f'{auth_server_url}/realms/{auth_realm}/protocol/openid-connect/token',
                      headers={'content-type': 'application/x-www-form-urlencoded'},
                      data={'grant_type': 'password',
                            'client_id': auth_client_id,
                            'username': auth_username,
                            'password': auth_password})
    r.raise_for_status()
    return r.json().get('access_token')

In [10]:
def secure_profiles_search(end_date: str, multi_search_value: str | None = None, limit: int = 100, offset: int = 0, createdBy: str | None = None, state: str | None = None) -> list:
    token = get_keycloak_token()
    r = requests.get(f"{snowprofiler_api_url}/secure/profiles/search", params={"endDate": end_date, "multiSearchValue": multi_search_value, "limit": limit, "offset": offset, "createdBy": createdBy, "state": state}, headers={'accept': 'application/json', 'authorization': f'Bearer {token}'})
    r.raise_for_status()
    return r.json()

def secure_profiles_search_extended_detailed(start_date: str, end_date: str, maxSlopeAngle: int | None = None) -> dict:
    token = get_keycloak_token()
    r = requests.get(f"{snowprofiler_api_url}/secure/profiles/search/extended/detailed", params={"startDate": start_date, "endDate": end_date, "maxSlopeAngle": maxSlopeAngle}, headers={'accept': 'application/json', 'authorization': f'Bearer {token}'})
    r.raise_for_status()
    return r.json()

def export_json(profile_id: str) -> str:
    r = requests.get(f"{snowprofiler_api_url}/public/profiles/{profile_id}/")
    r.raise_for_status()
    return r.text

def export_caaml(profile_id: str) -> str:
    r = requests.get(f"{snowprofiler_api_url}/public/profiles/{profile_id}/export/caaml?timezone=Europe/Zurich&languageCode=en")
    r.raise_for_status()
    return r.text

def write_caaml_to_file(profile_id: str, filename: str):
    with open(filename, "w") as f:
        f.write(export_caaml(profile_id))

In [11]:
profiles = secure_profiles_search(end_date="2024-08-31T23:59:59Z")

In [12]:
print(len(profiles))
pprint.pprint(profiles[0])

100
{'aspect': None,
 'canEdit': False,
 'coordinates': [9.809176, 46.82946],
 'date': '2024-07-04T09:00:00Z',
 'elevation': 2536,
 'id': '76f2e9f3-e259-4ae6-bf6b-82ef5153233f',
 'includedProfiles': {'avalancheReleaseProfiles': 0,
                      'compressionTests': 0,
                      'densityProfiles': 1,
                      'extendedColumnTests': 0,
                      'ramResistanceProfiles': 0,
                      'rutschblockTests': 0,
                      'stratigraphyProfiles': 0,
                      'temperatureProfiles': 1},
 'isFlat': True,
 'locationName': 'Davos Weissfluhjoch / Versuchsfeld - 5WJ-R, SnowTinel '
                 'project',
 'observers': [{'name': 'Bavay'}],
 'slopeAngle': 0,
 'state': 'PUBLISHED'}


In [13]:
profiles = secure_profiles_search(end_date="2024-08-31T23:59:59Z", multi_search_value="Bavay")

In [14]:
print(len(profiles))

54


In [15]:
profile_headers = secure_profiles_search_extended_detailed("2023-09-01T00:00:00Z", "2024-08-31T23:59:59Z")

In [16]:
print(len(profile_headers))

1072


In [17]:
write_caaml_to_file(profile_headers[0]["id"], "test.caaml.xml")

In [18]:
def read_ids_from_file(filename: str) -> list[str]:
    with open(filename, "r") as f:
        return [line.strip() for line in f.readlines()]

ids = read_ids_from_file("profiles/anrissprofile.csv")

print(len(ids))

for id in ids:
    write_caaml_to_file(id, f"profiles/anrissprofile/{id}.caaml.xml")

FileNotFoundError: [Errno 2] No such file or directory: 'profiles/anrissprofile.csv'